---
Initial Exploratery Data analysis 

In this sectin, we explore the raw, uncleaned dataset before any preprocessing. The goal is to understand the structure of the data, identfy potential quality issue, and form hypotheses about the features. These findings will directly motivate the preprocessing steps in Section 3.

We will:
1. Load and inspect the raw data
2. Check for missing values
3.Examine descriptive statistics
4.Explore the target variable distribution
5.Visualize features distributions and spot data quality issues
6.Explore categerical features
7.Examine corrlations in the raw data

### 2.1 — Load and Inspect the Raw Dataset

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:

df_raw = pd.read_csv('cardio_project_dataset.csv')
print('Dataset shape:', df_raw.shape)
print()
print(df_raw.head())

In [ ]:

print(df_raw.info())

**Observation:**  
The dataset contains **1200 rows and 12 columns**. All columns are numeric — no string or object types. There are **no null values** anywhere in the dataset. However, several things already stand out: the `age` column stores values in the tens of thousands, suggesting it is stored in **days** rather than years. The `gender`, `cholesterol`, `gluc`, `smoke`, `alco`, `active`, and `cardio` columns are categorical features stored as integers.

### 2.2 — Missing Values Check

In [ ]:

print("Missing values per column:")
print(df_raw.isnull().sum())
print()
print(f"Total missing values: {df_raw.isnull().sum().sum()}")

**Observation:**  
There are **no missing values** 

### 2.3 — Descriptive Statistics

In [ ]:
print(df_raw.describe().round(2))

**Observation:**  
Several issues are immediately visible from the statistics:
- **`age`**: Values range from ~14,319 to ~23,687 — this is age stored in **days**. Converting to years gives a range of roughly 39–65, which makes medical sense.
- **`ap_hi` (Systolic BP)**: The minimum is **-120** and the maximum is **907** — both are medically impossible. Normal systolic BP ranges from about 60 to 240 mmHg.
- **`ap_lo` (Diastolic BP)**: The minimum is **0** and the maximum is **1200** — similarly impossible. Normal diastolic BP is roughly 40 to 160 mmHg.
- **`weight`**: Minimum is **10 kg** — unrealistically low for an adult patient.
- **`gender`**: Encoded as 1 and 2. We will remap to 0 and 1 for consistency.

These observations directly motivate the outlier removal steps in the preprocessing section.

### 2.4 — Target Variable Distribution (cardio)

In [ ]:

cardio_counts = df_raw["cardio"].value_counts().sort_index()
cardio_labels = ["No Disease (0)", "Has Disease (1)"]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Bar chart
axes[0].bar(cardio_labels, cardio_counts.values,
            color=["green", "red"], edgecolor="white", width=0.5)
axes[0].set_title("Cardiovascular Disease — Count")
axes[0].set_ylabel("Number of Patients")
for i, v in enumerate(cardio_counts.values):
    axes[0].text(i, v + 5, str(v), ha="center", fontweight="bold")

# Pie chart
axes[1].pie(cardio_counts.values, labels=cardio_labels,
            colors=["green", "red"], autopct="%1.1f%%",
            startangle=90, wedgeprops={"edgecolor": "white"})
axes[1].set_title("Cardiovascular Disease — Proportion")

plt.suptitle("Target Variable Distribution: Cardiovascular Disease", fontsize=14)
plt.tight_layout()
plt.show()

print("Class distribution:")
print(cardio_counts)
print(f"\nBalance ratio: {cardio_counts.min() / cardio_counts.max():.2f}")

**Observation:**  
The dataset is **nearly perfectly balanced** — 607 patients without cardiovascular disease (50.6%) and 593 with it (49.4%). A balance ratio of ~0.98 means no class imbalance problems. This is excellent for clustering and classification tasks — no oversampling or undersampling will be needed.

### 2.5 — Age Distribution (Raw — Days vs Converted Years)

In [ ]:
# Show age distribution in raw format (days) vs converted to years
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Histogram of raw age in days
axes[0].hist(df_raw["age"], bins=20, color="steelblue", edgecolor="white")
axes[0].set_title("Age Distribution (Raw — in Days)")
axes[0].set_xlabel("Age (days)")
axes[0].set_ylabel("Number of Patients")

# Convert to years for comparison
age_years = (df_raw["age"] / 365).round(1)
axes[1].hist(age_years, bins=20, color="steelblue", edgecolor="white")
axes[1].set_title("Age Distribution (Converted — in Years)")
axes[1].set_xlabel("Age (years)")
axes[1].set_ylabel("Number of Patients")

plt.suptitle("Age: Raw (Days) vs Converted (Years)", fontsize=14)
plt.tight_layout()
plt.show()

print("Raw age stats (days):")
print(df_raw["age"].describe())
print()
print("Converted age stats (years):")
print(age_years.describe())

**Observation:**  
The raw `age` column stores values in **days** (ranging from ~14,319 to ~23,687). After converting by dividing by 365, we get ages ranging from approximately **39 to 65 years**, which is realistic for a cardiovascular disease study. The distribution is roughly bell-shaped, peaking around 53–55 years. **This confirms that age must be converted to years in preprocessing.**

### 2.6 — Blood Pressure: Distributions and Outlier Detection

In [ ]:
# Visualize blood pressure distributions and highlight invalid ranges
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# Histogram ap_hi
axes[0][0].hist(df_raw["ap_hi"], bins=30, color="steelblue", edgecolor="white")
axes[0][0].set_title("Systolic BP (ap_hi) — Histogram")
axes[0][0].set_xlabel("ap_hi (mmHg)")
axes[0][0].set_ylabel("Count")
axes[0][0].axvline(60,  color="red",    linestyle="--", label="Min valid (60)")
axes[0][0].axvline(240, color="orange", linestyle="--", label="Max valid (240)")
axes[0][0].legend()

# Histogram ap_lo
axes[0][1].hist(df_raw["ap_lo"], bins=30, color="steelblue", edgecolor="white")
axes[0][1].set_title("Diastolic BP (ap_lo) — Histogram")
axes[0][1].set_xlabel("ap_lo (mmHg)")
axes[0][1].set_ylabel("Count")
axes[0][1].axvline(40,  color="red",    linestyle="--", label="Min valid (40)")
axes[0][1].axvline(160, color="orange", linestyle="--", label="Max valid (160)")
axes[0][1].legend()

# Boxplot ap_hi
axes[1][0].boxplot(df_raw["ap_hi"])
axes[1][0].set_title("Systolic BP (ap_hi) — Boxplot")
axes[1][0].set_ylabel("ap_hi (mmHg)")

# Boxplot ap_lo
axes[1][1].boxplot(df_raw["ap_lo"])
axes[1][1].set_title("Diastolic BP (ap_lo) — Boxplot")
axes[1][1].set_ylabel("ap_lo (mmHg)")

plt.suptitle("Blood Pressure: Raw Distributions with Outliers", fontsize=14)
plt.tight_layout()
plt.show()

print("Invalid ap_hi values (< 60 or > 240):",
      len(df_raw[(df_raw["ap_hi"] < 60) | (df_raw["ap_hi"] > 240)]))
print("Invalid ap_lo values (< 40 or > 160):",
      len(df_raw[(df_raw["ap_lo"] < 40) | (df_raw["ap_lo"] > 160)]))
print("Invalid weight values (< 30 kg):",
      len(df_raw[df_raw["weight"] < 30]))

**Observation:**  
The histograms and boxplots clearly reveal **extreme outliers** in both blood pressure columns:
- **`ap_hi`**: Contains values as low as **-120** and as high as **907** mmHg — both physically impossible.
- **`ap_lo`**: Contains values of **0** and **1200** mmHg — also clearly invalid.
- **`weight`**: A minimum of **10 kg** is unrealistic for an adult.

The dashed lines mark the medically valid boundaries. The boxplots show extreme outliers far above the whiskers. These are most likely **data entry errors** and will be removed in preprocessing.

### 2.7 — Categorical Feature Distributions

In [ ]:
# Explore all categorical and binary features
fig, axes = plt.subplots(2, 3, figsize=(15, 9))

# Gender
gender_counts = df_raw["gender"].value_counts().sort_index()
axes[0][0].bar(["Female (1)", "Male (2)"], gender_counts.values,
               color=["orchid", "steelblue"], edgecolor="white")
axes[0][0].set_title("Gender Distribution")
axes[0][0].set_ylabel("Number of Patients")
for i, v in enumerate(gender_counts.values):
    axes[0][0].text(i, v + 3, str(v), ha="center", fontweight="bold")

# Cholesterol
chol_counts = df_raw["cholesterol"].value_counts().sort_index()
axes[0][1].bar(["Normal (1)", "Above Normal (2)", "Well Above (3)"],
               chol_counts.values, color=["lightgreen", "orange", "red"], edgecolor="white")
axes[0][1].set_title("Cholesterol Levels")
axes[0][1].set_ylabel("Number of Patients")
for i, v in enumerate(chol_counts.values):
    axes[0][1].text(i, v + 3, str(v), ha="center", fontweight="bold")

# Glucose
gluc_counts = df_raw["gluc"].value_counts().sort_index()
axes[0][2].bar(["Normal (1)", "Above Normal (2)", "Well Above (3)"],
               gluc_counts.values, color=["lightgreen", "orange", "red"], edgecolor="white")
axes[0][2].set_title("Glucose Levels")
axes[0][2].set_ylabel("Number of Patients")
for i, v in enumerate(gluc_counts.values):
    axes[0][2].text(i, v + 3, str(v), ha="center", fontweight="bold")

# Smoke
smoke_counts = df_raw["smoke"].value_counts().sort_index()
axes[1][0].bar(["Non-Smoker (0)", "Smoker (1)"], smoke_counts.values,
               color=["steelblue", "salmon"], edgecolor="white")
axes[1][0].set_title("Smoking Status")
axes[1][0].set_ylabel("Number of Patients")
for i, v in enumerate(smoke_counts.values):
    axes[1][0].text(i, v + 3, str(v), ha="center", fontweight="bold")

# Alcohol
alco_counts = df_raw["alco"].value_counts().sort_index()
axes[1][1].bar(["No Alcohol (0)", "Drinks (1)"], alco_counts.values,
               color=["steelblue", "salmon"], edgecolor="white")
axes[1][1].set_title("Alcohol Intake")
axes[1][1].set_ylabel("Number of Patients")
for i, v in enumerate(alco_counts.values):
    axes[1][1].text(i, v + 3, str(v), ha="center", fontweight="bold")

# Active
active_counts = df_raw["active"].value_counts().sort_index()
axes[1][2].bar(["Not Active (0)", "Active (1)"], active_counts.values,
               color=["salmon", "steelblue"], edgecolor="white")
axes[1][2].set_title("Physical Activity")
axes[1][2].set_ylabel("Number of Patients")
for i, v in enumerate(active_counts.values):
    axes[1][2].text(i, v + 3, str(v), ha="center", fontweight="bold")

plt.suptitle("Categorical Feature Distributions (Raw Data)", fontsize=14)
plt.tight_layout()
plt.show()

**Observation:**  
- **Gender:** More female patients (coded as 1) than male patients (coded as 2). The encoding will be remapped to 0/1 in preprocessing for consistency.
- **Cholesterol:** The majority of patients have Normal cholesterol (1). This is an ordinal feature — the encoding already reflects the natural ordering.
- **Glucose:** Most patients have Normal glucose. A notable portion falls in the Well Above Normal category, which may indicate diabetic patients.
- **Smoking & Alcohol:** Both are heavily imbalanced — most patients are non-smokers and non-drinkers. This could reduce the predictive power of these features.
- **Physical Activity:** About 79% of patients report being physically active.

### 2.8 — Height and Weight Distributions

In [ ]:
# Distribution of height and weight
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].hist(df_raw["height"], bins=25, color="steelblue", edgecolor="white")
axes[0].set_title("Height Distribution (cm)")
axes[0].set_xlabel("Height (cm)")
axes[0].set_ylabel("Number of Patients")

axes[1].hist(df_raw["weight"], bins=25, color="steelblue", edgecolor="white")
axes[1].set_title("Weight Distribution (kg)")
axes[1].set_xlabel("Weight (kg)")
axes[1].set_ylabel("Number of Patients")
axes[1].axvline(30, color="red", linestyle="--", label="Min valid (30 kg)")
axes[1].legend()

plt.suptitle("Height and Weight Distributions (Raw Data)", fontsize=14)
plt.tight_layout()
plt.show()

print("Height stats:")
print(df_raw["height"].describe())
print()
print("Weight stats:")
print(df_raw["weight"].describe())

**Observation:**  
- **Height:** Ranges from about 55 to 207 cm with a mean around 164 cm. The distribution is roughly normal with no clearly invalid values — this column looks clean.
- **Weight:** Mostly between 60–90 kg, with a right skew. However, the minimum is **10 kg**, which is clearly invalid for an adult. The red dashed line marks the 30 kg threshold we will use to filter out invalid records in preprocessing. Also, since we have both height and weight, we will engineer a **BMI** feature in preprocessing.

### 2.9 — Correlation Heatmap (Raw Data)

In [ ]:
# Correlation heatmap on raw data
plt.figure(figsize=(11, 8))

corr_raw = df_raw.corr()

sns.heatmap(corr_raw,
            annot=True,
            fmt=".2f",
            cmap="coolwarm",
            center=0,
            square=True,
            linewidths=0.5)

plt.title("Correlation Heatmap — Raw Data", fontsize=14, pad=15)
plt.tight_layout()
plt.show()

**Observation:**  
Even on raw data, several correlations are already visible:
- **`ap_hi` and `ap_lo`** have a **strong positive correlation (~0.73)** — expected since both measure blood pressure.
- **`age` and `cardio`** show a **moderate positive correlation (~0.24)** — older patients tend to have cardiovascular disease.
- **`cholesterol` and `cardio`** show a positive correlation — higher cholesterol is linked to disease.
- **`smoke` and `gender`** have a moderate positive correlation — males in this dataset tend to smoke more.

Note: `age` is still in days here, so its scale differs greatly from other features. Correlations will be cleaner after converting age and removing outliers in preprocessing.

### Initial EDA Summary

From exploring the raw dataset, we identified the following key findings that directly motivate our preprocessing steps:

| # | Finding | Preprocessing Action |
|---|---------|---------------------|
| 1 | `age` stored in days (14,319 – 23,687) | Convert to years (÷ 365) |
| 2 | `ap_hi` has impossible values (e.g. -120, 907) | Remove rows outside 60–240 mmHg |
| 3 | `ap_lo` has impossible values (e.g. 0, 1200) | Remove rows outside 40–160 mmHg |
| 4 | `weight` minimum is 10 kg | Remove rows below 30 kg |
| 5 | No missing values in any column | No imputation needed |
| 6 | Target `cardio` is nearly balanced (50.6% / 49.4%) | No class balancing needed |
| 7 | `gender` uses values 1/2 instead of 0/1 | Remap to 0 = Female, 1 = Male |
| 8 | Height and weight available but no BMI column | Engineer BMI = weight / (height/100)² |
| 9 | Features have very different scales | Apply StandardScaler on continuous columns |

All of these findings are addressed in **Section 3 — Data Preprocessing**.